# RAG (Retrieval-Augmented Generation) untuk Asisten Legal Tim
## Submission Proyek Akhir - PGABL

| **Informasi** | **Detail** |
|---|---|
| **Nama** | Faishal Anwar Hasyim |
| **Email** | anwarfaishal86@gmail.com |

Notebook ini membangun sistem RAG menggunakan model fine-tuned dan 4 dokumen PDF Undang-Undang.

---


## 1. Instalasi Library

In [1]:
# === Suppress ALL warnings including jupyter_client DeprecationWarning ===
import warnings
warnings.filterwarnings("ignore")

# Override showwarning to suppress kernel-level DeprecationWarnings (jupyter_client/session.py)
warnings.showwarning = lambda *args, **kwargs: None

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("chromadb").setLevel(logging.ERROR)
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"

# Install dependencies
!pip install unsloth[colab-new] -q
!pip install --no-deps unsloth[colab-no-deps] -q
!pip install langchain langchain-classic langchain-community langchain-huggingface langchain-text-splitters -q
!pip install sentence-transformers faiss-cpu chromadb -q
!pip install pypdf rank-bm25 duckduckgo-search -q
!pip install gradio gdown -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 118.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 80.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

## 2. Import Library dan Setup

In [2]:
import warnings
warnings.filterwarnings("ignore")
warnings.showwarning = lambda *args, **kwargs: None

import os, torch, re, getpass
from IPython.display import display, Markdown

def get_secret(key):
    """Ambil secret dari Colab Secrets, env var, atau input manual."""
    try:
        from google.colab import userdata
        return userdata.get(key)
    except Exception:
        val = os.environ.get(key)
        if val:
            return val
        return getpass.getpass(f"Masukkan {key}: ")

os.environ["HF_TOKEN"] = get_secret("HF_TOKEN")
print("Libraries imported successfully!")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Libraries imported successfully!
CUDA available: True
GPU: Tesla T4


## 3. Download dan Memuat Dokumen PDF

Mengunduh 4 file PDF Undang-Undang yang WAJIB digunakan seluruhnya.

In [3]:
# Download dokumen PDF dari Google Drive
import gdown
import glob
import os

os.makedirs("documents", exist_ok=True)

# Cek apakah PDF sudah ada di folder documents
pdf_files = glob.glob("documents/**/*.pdf", recursive=True)

if pdf_files:
    print(f"PDF sudah tersedia ({len(pdf_files)} file), skip download.")
else:
    print("Downloading PDF dari Google Drive...")
    folder_url = "https://drive.google.com/drive/folders/1Vw6Jh0mcJ4NAqkYrap3eynOAnTHwkwyB"
    gdown.download_folder(folder_url, output="documents", quiet=False, use_cookies=False, remaining_ok=True)

# Verifikasi hasil
pdf_files = glob.glob("documents/**/*.pdf", recursive=True)
print(f"\nIsi folder documents:")
for root, dirs, files_list in os.walk("documents"):
    for file in files_list:
        print(f"  {os.path.join(root, file)}")
print(f"\nJumlah PDF: {len(pdf_files)}")
for f in pdf_files:
    print(f"  ✅ {f}")

assert len(pdf_files) >= 3, "❌ PDF tidak ditemukan. Pastikan folder Google Drive di-share: Anyone with the link."
print("\n✅ Dokumen siap untuk diproses!")

Retrieving folder contents


Processing file 1F29FFzmYEOquClljAB17Aqm_u_TYBcsK PP Nomor 5 Tahun 2021.pdf
Processing file 1KtTKAk8WJe_Wc_YTNsjrKYHrQ1_v9YTh PP Nomor 35 Tahun 2021.pdf
Processing file 1ptNLV2Rc0_aj4M3bR1fsjMnhArwZhF42 PP Nomor 51 Tahun 2023.pdf
Processing file 1Z39hYwbtpPYRUqrrJjgrOGRkSVsKjHor UU Nomor 6 Tahun 2023.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1F29FFzmYEOquClljAB17Aqm_u_TYBcsK
To: /content/documents/PP Nomor 5 Tahun 2021.pdf
100%|██████████| 17.1M/17.1M [00:00<00:00, 35.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KtTKAk8WJe_Wc_YTNsjrKYHrQ1_v9YTh
To: /content/documents/PP Nomor 35 Tahun 2021.pdf
100%|██████████| 2.52M/2.52M [00:00<00:00, 21.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ptNLV2Rc0_aj4M3bR1fsjMnhArwZhF42
To: /content/documents/PP Nomor 51 Tahun 2023.pdf
100%|██████████| 2.77M/2.77M [00:00<00:00, 165MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Z39hYwbtpPYRUqrrJjgrOGRkSVsKjHor
To: /content/documents/UU Nomor 6 Tahun 2023.pdf
100%|██████████| 85.4M/85.4M [00:01<00:00, 55.2MB/s]


Isi folder documents:
  documents/UU Nomor 6 Tahun 2023.pdf
  documents/PP Nomor 35 Tahun 2021.pdf
  documents/PP Nomor 5 Tahun 2021.pdf
  documents/PP Nomor 51 Tahun 2023.pdf

Jumlah PDF: 4
  ✅ documents/UU Nomor 6 Tahun 2023.pdf
  ✅ documents/PP Nomor 35 Tahun 2021.pdf
  ✅ documents/PP Nomor 5 Tahun 2021.pdf
  ✅ documents/PP Nomor 51 Tahun 2023.pdf

✅ Dokumen siap untuk diproses!



Download completed


## 4. Memuat dan Memproses Dokumen PDF dengan Metadata Enrichment (Skilled)

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Memuat semua PDF
all_documents = []
for pdf_path in sorted(pdf_files):
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    # Metadata Enrichment (Skilled): Tambahkan metadata ke setiap halaman
    for doc in docs:
        filename = os.path.basename(pdf_path)
        doc.metadata["source_file"] = filename
        doc.metadata["doc_type"] = "Undang-Undang"

        # Deteksi nama UU dari filename
        if "cipta_kerja" in filename.lower() or "ciptakerja" in filename.lower():
            doc.metadata["uu_name"] = "UU Cipta Kerja"
            doc.metadata["category"] = "ketenagakerjaan"
        elif "ketenagakerjaan" in filename.lower() or "13_2003" in filename.lower():
            doc.metadata["uu_name"] = "UU Ketenagakerjaan"
            doc.metadata["category"] = "ketenagakerjaan"
        elif "ppn" in filename.lower() or "pajak_pertambahan" in filename.lower():
            doc.metadata["uu_name"] = "UU PPN"
            doc.metadata["category"] = "perpajakan"
        elif "pph" in filename.lower() or "pajak_penghasilan" in filename.lower():
            doc.metadata["uu_name"] = "UU PPh"
            doc.metadata["category"] = "perpajakan"
        else:
            doc.metadata["uu_name"] = filename
            doc.metadata["category"] = "umum"

    all_documents.extend(docs)

print(f"Total halaman dari semua PDF: {len(all_documents)}")
if all_documents:
    print(f"\nMetadata contoh:")
    print(all_documents[0].metadata)
else:
    print("\n⚠️ Tidak ada dokumen dimuat. Upload PDF secara manual ke folder documents/")

Total halaman dari semua PDF: 1949

Metadata contoh:
{'producer': '', 'creator': 'Canon', 'creationdate': '2021-02-18T15:54:05+07:00', 'moddate': '2021-02-18T16:07:05+07:00', 'source': 'documents/PP Nomor 35 Tahun 2021.pdf', 'total_pages': 56, 'page': 0, 'page_label': '1', 'source_file': 'PP Nomor 35 Tahun 2021.pdf', 'doc_type': 'Undang-Undang', 'uu_name': 'PP Nomor 35 Tahun 2021.pdf', 'category': 'umum'}


## 5. Text Splitting - Parent-Child Chunks (Skilled)

Membangun Parent-Child Retriever:
- **Child Chunks**: potongan kecil untuk pencarian vektor
- **Parent Chunks**: potongan besar untuk konteks LLM

In [5]:
# Parent Chunks (potongan besar untuk konteks LLM)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=["\n\n", "\n", "Pasal ", "Ayat ", "BAB ", ". ", " "],
)

# Child Chunks (potongan kecil untuk pencarian vektor)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", "Pasal ", "Ayat ", ". ", " "],
)

# Buat parent chunks
parent_chunks = parent_splitter.split_documents(all_documents)

# Buat child chunks dari parent chunks
all_child_chunks = []
for i, parent in enumerate(parent_chunks):
    children = child_splitter.split_documents([parent])
    for child in children:
        child.metadata["parent_id"] = i
        child.metadata["parent_content"] = parent.page_content
    all_child_chunks.extend(children)

print(f"Jumlah parent chunks: {len(parent_chunks)}")
print(f"Jumlah child chunks: {len(all_child_chunks)}")
print(f"\nContoh child chunk metadata keys: {list(all_child_chunks[0].metadata.keys())}")

Jumlah parent chunks: 1962
Jumlah child chunks: 5337

Contoh child chunk metadata keys: ['producer', 'creator', 'creationdate', 'moddate', 'source', 'total_pages', 'page', 'page_label', 'source_file', 'doc_type', 'uu_name', 'category', 'parent_id', 'parent_content']


## 6. Embedding dengan Model Open-Source dan Vector Database

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Model embedding open-source
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
)

print("Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2")

# Simpan child chunks ke FAISS Vector Database
# Perlu menghapus parent_content dari metadata sebelum indexing (terlalu besar)
child_docs_for_faiss = []
for doc in all_child_chunks:
    import copy
    new_doc = copy.deepcopy(doc)
    if "parent_content" in new_doc.metadata:
        del new_doc.metadata["parent_content"]
    child_docs_for_faiss.append(new_doc)

vectorstore = FAISS.from_documents(child_docs_for_faiss, embedding_model)

print(f"FAISS Vector Database dibuat dengan {len(child_docs_for_faiss)} child chunks")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
FAISS Vector Database dibuat dengan 5337 child chunks


## 7. Membangun Ensemble Retriever - BM25 + Semantic (Skilled)

In [7]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# Hybrid Retriever: FAISS (semantic) + BM25 (keyword)
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# BM25 menggunakan child chunks
bm25_retriever = BM25Retriever.from_documents(child_docs_for_faiss, k=10)

# Ensemble: 60% semantic + 40% keyword
ensemble_retriever = EnsembleRetriever(
    retrievers=[faiss_retriever, bm25_retriever],
    weights=[0.6, 0.4]
)

print("Hybrid Retriever (FAISS + BM25) siap!")

# Multi-Query Expansion
def expand_query(question):
    """Expand query menjadi beberapa versi untuk meningkatkan recall."""
    queries = [question]

    # Versi lebih formal
    formal = question.replace("saya", "pekerja").replace("dapat", "berhak memperoleh")
    queries.append(formal)

    # Versi dengan keyword hukum
    legal_keywords = question + " undang-undang peraturan pasal ketentuan"
    queries.append(legal_keywords)

    return queries

# HyDE: Hypothetical Document Embedding
def generate_hyde_doc(question, model_for_hyde, tokenizer_for_hyde):
    """Generate hypothetical document sebagai query embedding."""
    hyde_prompt = f"""Tuliskan sebuah paragraf dari undang-undang Indonesia yang menjawab pertanyaan berikut:
Pertanyaan: {question}
Paragraf undang-undang:"""

    messages = [
        {"role": "system", "content": "Anda adalah ahli hukum Indonesia."},
        {"role": "user", "content": hyde_prompt},
    ]

    inputs = tokenizer_for_hyde.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model_for_hyde.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7)
    hyde_doc = tokenizer_for_hyde.decode(outputs[0], skip_special_tokens=True)
    return hyde_doc

print("Multi-Query + HyDE functions ready!")

Hybrid Retriever (FAISS + BM25) siap!
Multi-Query + HyDE functions ready!


## 8. Pengujian Sistem Retrieval

In [8]:
from sentence_transformers import CrossEncoder

# Reranker Model
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_documents(query, docs, top_k=5):
    """Rerank dokumen menggunakan CrossEncoder."""
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)

    # Pair dokumen dengan skor dan sort
    scored_docs = list(zip(docs, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    return scored_docs[:top_k]

# Test retrieval + reranking
test_query = "hak pekerja lembur"
retrieved = ensemble_retriever.invoke(test_query)
print(f"Retrieved {len(retrieved)} documents")

reranked = rerank_documents(test_query, retrieved)
print(f"\nTop-5 after reranking:")
for doc, score in reranked:
    print(f"  Score: {score:.4f} | {doc.page_content[:80]}...")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Retrieved 20 documents

Top-5 after reranking:
  Score: 7.2719 | melaksanakan pekerjaan dalam Waktu Kerja Lembur.
Perjanjian Kerja adalah perjanj...
  Score: 5.2799 | yaitu pembayaran Upah Pekerja/Buruh didahulukan
dari semua jenis kreditur termas...
  Score: 5.0551 | PRESiDEN
REPUBLIK INDONESI,A.
-18-
(21 Perintah dan persetujuan sebagaimana dima...
  Score: 4.1691 | Pekerja/ Buruh maka Pekerja/Buruh berhak atas:
uang pesangon
Pasal 40 ayat (21;
...
  Score: 4.0068 | PRESIDEN
REPI.IBLIK INDONESIA
-549-
(21 Pengusaha yang mempekerjakan Pekerja/Bur...


## 9. Parent-Child Retrieval (Skilled)

Mengganti child chunk dengan parent chunk yang lebih besar untuk konteks LLM.

In [9]:
def get_parent_chunks(retrieved_child_docs, all_child_chunks):
    """Mengambil parent chunks dari child chunks yang di-retrieve."""
    parent_ids = set()
    parent_contents = {}

    for doc in retrieved_child_docs:
        parent_id = doc.metadata.get("parent_id")
        if parent_id is not None and parent_id not in parent_ids:
            parent_ids.add(parent_id)
            # Cari parent content dari child_chunks original
            for child in all_child_chunks:
                if child.metadata.get("parent_id") == parent_id:
                    parent_contents[parent_id] = {
                        "content": child.metadata.get("parent_content", doc.page_content),
                        "metadata": {k:v for k,v in child.metadata.items() if k != "parent_content"}
                    }
                    break

    return parent_contents

# Test Parent-Child Retrieval
parent_results = get_parent_chunks(retrieved, all_child_chunks)
print(f"Unique parent chunks retrieved: {len(parent_results)}")
for pid, pdata in list(parent_results.items())[:3]:
    print(f"\n--- Parent {pid} ---")
    print(f"Metadata: {pdata['metadata']}")
    print(f"Content (300 chars): {pdata['content'][:300]}...")

Unique parent chunks retrieved: 17

--- Parent 2 ---
Metadata: {'producer': '', 'creator': 'Canon', 'creationdate': '2021-02-18T15:54:05+07:00', 'moddate': '2021-02-18T16:07:05+07:00', 'source': 'documents/PP Nomor 35 Tahun 2021.pdf', 'total_pages': 56, 'page': 2, 'page_label': '3', 'source_file': 'PP Nomor 35 Tahun 2021.pdf', 'doc_type': 'Undang-Undang', 'uu_name': 'PP Nomor 35 Tahun 2021.pdf', 'category': 'umum', 'parent_id': 2}
Content (300 chars): PRESIDEN
REPUBLIK INDONESIA
-3-
5
b. usaha-usaha sosial dan usaha-usaha lain yang
mempunyai pengurus dan mempekerjakan orang
lain dengan membayar Upah atau imbalan dalam
bentuk lain.
Serikat Pekerja/Serikat Buruh adalah organisasi yang
dibentuk dari, oleh, dan untuk Pekerja/Buruh baik di
Perusahaan ...

--- Parent 22 ---
Metadata: {'producer': '', 'creator': 'Canon', 'creationdate': '2021-02-18T15:54:05+07:00', 'moddate': '2021-02-18T16:07:05+07:00', 'source': 'documents/PP Nomor 35 Tahun 2021.pdf', 'total_pages': 56, 'page': 22, 'page_la

## 10. HyDE - Hypothetical Document Embeddings (Advanced)

HyDE menggunakan **model LLM** untuk men-generate dokumen hipotesis secara dinamis.
Model menghasilkan minimal **2 dokumen hipotesis** (`n_hypotheses=2`) yang berbeda,
kemudian di-embed bersama query asli dan di-average untuk retrieval yang lebih akurat.


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

def generate_hypothetical_documents(query, model, tokenizer, n_hypotheses=2):
    """
    Generate hypothetical documents (HyDE) secara dinamis menggunakan model LLM.
    Model menghasilkan n_hypotheses dokumen hipotesis berbasis query.
    Setiap hipotesis di-generate secara independen dengan variasi prompt.
    """
    hypotheses = []

    for i in range(n_hypotheses):
        # Variasi system prompt agar hipotesis beragam
        if i == 0:
            system_msg = (
                "Anda adalah pakar hukum ketenagakerjaan Indonesia. "
                "Tuliskan sebuah paragraf jawaban yang relevan dan detail "
                "untuk pertanyaan berikut berdasarkan peraturan perundang-undangan Indonesia."
            )
        else:
            system_msg = (
                "Anda adalah konsultan hukum Indonesia. "
                "Berikan penjelasan singkat tentang ketentuan hukum yang relevan "
                "dengan pertanyaan berikut menurut undang-undang yang berlaku."
            )

        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": query},
        ]

        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")

        attention_mask = (inputs != tokenizer.pad_token_id).long()

        outputs = model.generate(
            input_ids=inputs,
            attention_mask=attention_mask,
            max_new_tokens=150,
            max_length=None,
            temperature=0.8 + (i * 0.1),  # Variasi temperatur per hipotesis
            top_p=0.9,
            do_sample=True,
            use_cache=True,
        )

        hypothesis = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
        hypotheses.append(hypothesis.strip())

    return hypotheses

def hyde_retrieve(query, vectorstore, embedding_model, n_hypotheses=2, top_k=5, model=None, tokenizer=None):
    """
    HyDE retrieval: generate hypothetical docs with LLM, embed query + docs, average embeddings, search.
    """
    # Generate hypothetical documents menggunakan LLM
    hypotheses = generate_hypothetical_documents(query, model, tokenizer, n_hypotheses)

    print(f"Generated {len(hypotheses)} hypothetical documents")
    for i, h in enumerate(hypotheses):
        print(f"  Hypothesis {i+1} (100 chars): {h[:100]}...")

    # Embed query dan hypothetical documents
    all_texts = [query] + hypotheses
    embeddings = embedding_model.embed_documents(all_texts)

    # Average embedding
    avg_embedding = np.mean(embeddings, axis=0).tolist()

    # Search dengan averaged embedding
    results = vectorstore.similarity_search_by_vector(avg_embedding, k=top_k)

    return results

# Catatan: Test HyDE akan dilakukan setelah model di-load pada section selanjutnya.
print("HyDE functions defined (LLM-based generation).")
print("Fungsi generate_hypothetical_documents() menggunakan model LLM untuk generate hipotesis secara dinamis.")
print("Test akan dijalankan setelah model dimuat.")


HyDE functions defined (LLM-based generation).
Fungsi generate_hypothetical_documents() menggunakan model LLM untuk generate hipotesis secara dinamis.
Test akan dijalankan setelah model dimuat.


## 11. Reranker - Cross-Encoder (Advanced)

Menggunakan model Reranker untuk mengurutkan ulang dan mengambil Top-K.

In [11]:
from sentence_transformers import CrossEncoder

# Load Cross-Encoder Reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_documents(query, documents, top_k=3, threshold=-5.0):
    """
    Rerank dokumen menggunakan Cross-Encoder dan ambil Top-K.
    Jika skor Top-1 di bawah threshold, return None untuk trigger web search.
    """
    if not documents:
        return [], None, True

    # Siapkan pairs untuk Cross-Encoder
    pairs = [[query, doc.page_content] for doc in documents]

    # Dapatkan skor
    scores = reranker.predict(pairs)

    # Gabungkan dokumen dengan skor
    scored_docs = list(zip(documents, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    # Ambil Top-K
    top_docs = scored_docs[:top_k]

    # Cek threshold Top-1
    top1_score = top_docs[0][1] if top_docs else 0
    below_threshold = top1_score < threshold

    return top_docs, top1_score, below_threshold

# Test Reranker
test_query = "Berapa jam maksimal lembur dalam sehari?"
test_docs = ensemble_retriever.invoke(test_query)
reranked, top1_score, below_thresh = rerank_documents(test_query, test_docs, top_k=3, threshold=-5.0)

print(f"Query: {test_query}")
print(f"Top-1 Relevance Score: {top1_score:.4f}")
print(f"Below threshold (-5.0): {below_thresh}")
print(f"\nTop-3 Reranked Documents:")
for doc, score in reranked:
    print(f"  Score: {score:.4f} | {doc.page_content[:100]}...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: Berapa jam maksimal lembur dalam sehari?
Top-1 Relevance Score: 4.7023
Below threshold (-5.0): False

Top-3 Reranked Documents:
  Score: 4.7023 | (1) Perusahaan yang mempekerjakan Pekerja/Buruh
selama Waktu Kerja Lembur berkewajiban:
a. membayar ...
  Score: 3.0935 | Pemerintah.
24. Ketentuan Pasal 78 diubah sehingga berbunyi sebagai
berikut:
Pasal 78
(1) Pengusaha ...
  Score: 2.8814 | 40 (empat puluh) jam seminggu, dengan ketentuan:
a. perhitungan Upah Kerja Lembur dilaksanakan
sebag...


## 12. DuckDuckGo Search Fallback (Advanced)

Jika skor Reranker Top-1 di bawah threshold, beralih ke web search.

In [12]:
from duckduckgo_search import DDGS

def duckduckgo_search(query, max_results=3):
    """Mencari informasi dari internet menggunakan DuckDuckGo."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query + " hukum Indonesia", max_results=max_results))

        search_context = ""
        for r in results:
            search_context += f"Sumber: {r['href']}\n"
            search_context += f"{r['body']}\n\n"

        return search_context.strip()
    except Exception as e:
        return f"Pencarian web gagal: {str(e)}"

# Test
test_web = duckduckgo_search("ketentuan lembur karyawan Indonesia")
print("DuckDuckGo Search Result (300 chars):")
print(test_web[:300])

DuckDuckGo Search Result (300 chars):
Sumber: https://support.google.com/youtubetv/?hl=en
Official YouTube TV Help Center where you can find tips and tutorials on using YouTube TV and other answers to frequently asked …

Sumber: https://support.google.com/youtube/answer/9288567?hl=ru
Используя YouTube, вы становитесь частью огромного со


## 13. Metadata Filtering dan Sitasi (Skilled)

Membuat metadata filtering serta sitasi pada jawaban AI.

In [13]:
def filter_by_metadata(documents, category=None, uu_name=None):
    """Filter dokumen berdasarkan metadata."""
    filtered = documents
    if category:
        filtered = [d for d in filtered if d.metadata.get("category", "").lower() == category.lower()]
    if uu_name:
        filtered = [d for d in filtered if uu_name.lower() in d.metadata.get("uu_name", "").lower()]
    return filtered if filtered else documents  # Fallback ke semua jika filter kosong

def generate_citations(documents):
    """Generate sitasi dari dokumen yang digunakan."""
    citations = []
    seen = set()
    for doc in documents:
        source = doc.metadata.get("source_file", "Unknown")
        page = doc.metadata.get("page", "N/A")
        uu = doc.metadata.get("uu_name", "Unknown")
        cite = f"{uu} (File: {source}, Halaman: {page})"
        if cite not in seen:
            seen.add(cite)
            citations.append(cite)
    return citations

# Test
test_docs_meta = ensemble_retriever.invoke("ketentuan lembur")
citations = generate_citations(test_docs_meta)
print("Sitasi:")
for c in citations:
    print(f"  - {c}")

Sitasi:
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 320)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 190)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 101)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 482)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 358)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 193)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 203)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 427)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 693)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 378)
  - PP Nomor 35 Tahun 2021.pdf (File: PP Nomor 35 Tahun 2021.pdf, Halaman: 17)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 558)
  - PP Nomor 35 Tahun 2021.pdf (File: PP Nomor 35 Tahun

## 14. Memuat Model Fine-tuned untuk Inferensi

In [14]:
import warnings
warnings.filterwarnings("ignore")
warnings.showwarning = lambda *args, **kwargs: None
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

max_seq_length = 2048
HF_USERNAME = get_secret("HF_USERNAME")

try:
    INFERENCE_MODEL = f"{HF_USERNAME}/qwen2.5-1.5b-pgabl-legal-grpo-faishal"
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=INFERENCE_MODEL, max_seq_length=max_seq_length,
        dtype=None, load_in_4bit=True,
    )
    print(f"Model GRPO loaded: {INFERENCE_MODEL}")
except:
    INFERENCE_MODEL = f"{HF_USERNAME}/qwen2.5-1.5b-pgabl-legal-sft-faishal"
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=INFERENCE_MODEL, max_seq_length=max_seq_length,
        dtype=None, load_in_4bit=True,
    )
    print(f"Model SFT loaded: {INFERENCE_MODEL}")

tokenizer = get_chat_template(tokenizer, chat_template="chatml")
FastLanguageModel.for_inference(model)
print("Model siap untuk inferensi!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model GRPO loaded: Faishal-Anwar/qwen2.5-1.5b-pgabl-legal-grpo-faishal


Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_6qa77rgm/tokenizer_config.json.


Model siap untuk inferensi!


In [15]:
# Test HyDE dengan model LLM yang sudah dimuat
hyde_query = "Bagaimana ketentuan waktu kerja lembur menurut peraturan pemerintah?"
hyde_results = hyde_retrieve(
    hyde_query, vectorstore, embedding_model,
    n_hypotheses=2, top_k=5, model=model, tokenizer=tokenizer
)

print(f"\nHyDE Retrieved {len(hyde_results)} documents:")
for i, doc in enumerate(hyde_results):
    print(f"  Doc {i+1}: {doc.metadata.get('source_file', 'N/A')} - {doc.page_content[:100]}...")


Generated 2 hypothetical documents
  Hypothesis 1 (100 chars): Ketentuan waktu kerja lembur adalah bentuk dari regulasi yang memperkirakan pengaturan kerja lembur ...
  Hypothesis 2 (100 chars): Waktu kerja lembur di Indonesia adalah waktu tambahan yang dibedakan untuk karyawan yang bekerja leb...

HyDE Retrieved 5 documents:
  Doc 1: UU Nomor 6 Tahun 2023.pdf - PRESIDEN
REPI.IBLIK INDONESIA
-549-
(21 Pengusaha yang mempekerjakan Pekerja/Buruh
melebihi waktu ke...
  Doc 2: UU Nomor 6 Tahun 2023.pdf - (3) Pelaksanaan ketentuan sebagaimana dimaksud
pada ayat (21 dan ayat (2a) diatur dengan atau
berdas...
  Doc 3: PP Nomor 5 Tahun 2021.pdf - kegiatan usaha penempatan pekerja migran Indonesia. 
(4) Sanksi administratif berupa peringatan tert...
  Doc 4: PP Nomor 35 Tahun 2021.pdf - PRESIDEN
REPUBLIK INDONESIA
-5-
17. Pengawas Ketenagakerjaan adalah pegawai negeri sipil
yang diberi...
  Doc 5: PP Nomor 35 Tahun 2021.pdf - PRESIDEN
REPUBLIK INDONESIA
-t7-
Bagian Ketiga
Waktu Kerja Lembur
Pas

## 15. Merangkai Pipeline RAG

Fungsi utama yang menggabungkan retrieval, reranking, dan generation.

In [22]:
import warnings
warnings.filterwarnings("ignore")
warnings.showwarning = lambda *args, **kwargs: None

def rag_pipeline(question, use_hyde=True, rerank_threshold=-5.0, top_k=3):
    """Pipeline RAG lengkap dengan HyDE, Reranking, dan DuckDuckGo fallback."""

    # Step 1: Retrieval
    if use_hyde:
        retrieved_docs = hyde_retrieve(question, vectorstore, embedding_model, n_hypotheses=2, top_k=5, model=model, tokenizer=tokenizer)
    else:
        retrieved_docs = ensemble_retriever.invoke(question)

    # Step 2: Reranking
    reranked_docs, top1_score, below_threshold = rerank_documents(
        question, retrieved_docs, top_k=top_k, threshold=rerank_threshold
    )

    # Step 3: Cek threshold - fallback ke web search jika perlu
    web_context = ""
    if below_threshold:
        print(f"[INFO] Skor Reranker Top-1 ({top1_score:.4f}) di bawah threshold. Beralih ke DuckDuckGo Search...")
        web_context = duckduckgo_search(question)

    # Step 4: Kumpulkan konteks (parent chunks jika tersedia)
    if not below_threshold:
        context_parts = []
        used_docs = [doc for doc, score in reranked_docs]
        parent_results = get_parent_chunks(used_docs, all_child_chunks)

        if parent_results:
            for pid, pdata in parent_results.items():
                context_parts.append(pdata["content"])
        else:
            for doc, score in reranked_docs:
                context_parts.append(doc.page_content)

        context = "\n\n".join(context_parts)
        citations = generate_citations(used_docs)
    else:
        context = web_context
        citations = ["Sumber: DuckDuckGo Web Search"]

    # Step 5: Generate jawaban
    system_prompt = (
        "Anda adalah asisten AI legal yang membantu Tim Legal perusahaan. "
        "Jawablah pertanyaan berdasarkan konteks dokumen yang diberikan. "
        "Jika informasi tidak tersedia dalam konteks, katakan bahwa Anda tidak menemukan informasi tersebut. "
        "Jawab dalam Bahasa Indonesia dengan jelas dan akurat."
    )

    user_prompt = f"Berdasarkan konteks dokumen berikut:\n\n{context}\n\nPertanyaan: {question}\n\nJawaban:"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    attention_mask = (inputs != tokenizer.pad_token_id).long()

    outputs = model.generate(
        input_ids=inputs,
        attention_mask=attention_mask,
        max_new_tokens=1024,
        max_length=None,
        min_new_tokens=30,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    # Step 6: Tambahkan sitasi
    citation_text = "\n".join([f"  - {c}" for c in citations])
    full_response = f"{response}\n\n---\n**Sumber Referensi:**\n{citation_text}"

    return full_response, top1_score

print("RAG Pipeline berhasil dibangun!")

RAG Pipeline berhasil dibangun!


## 16. Test RAG Pipeline

In [23]:
import warnings
warnings.filterwarnings("ignore")
warnings.showwarning = lambda *args, **kwargs: None

test_question = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"

print(f"Question: {test_question}")
print("=" * 60)

answer, score = rag_pipeline(test_question)
print(f"\nReranker Top-1 Score: {score:.4f}")
print(f"\nJawaban:")
display(Markdown(answer))

Question: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?
Generated 2 hypothetical documents
  Hypothesis 1 (100 chars): Ya, Anda berhak untuk uang lembur yang ditentukan oleh pemerintah dalam peraturan perundang-undangan...
  Hypothesis 2 (100 chars): Tentu, anda berhak untuk mengklaim uang lembur yang telah anda habiskan beresin laporan. Uang lembur...

Reranker Top-1 Score: -3.1949

Jawaban:


Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

---
**Sumber Referensi:**
  - PP Nomor 5 Tahun 2021.pdf (File: PP Nomor 5 Tahun 2021.pdf, Halaman: 215)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 558)
  - PP Nomor 35 Tahun 2021.pdf (File: PP Nomor 35 Tahun 2021.pdf, Halaman: 20)

In [27]:
import warnings
warnings.filterwarnings("ignore")
warnings.showwarning = lambda *args, **kwargs: None

test_questions = [
    "Apa saja hak pekerja yang di-PHK menurut undang-undang?",
    "Berapa lama masa percobaan kerja yang diperbolehkan?",
    "Bagaimana ketentuan pajak penghasilan untuk karyawan?",
]

for q in test_questions:
    print(f"\n" + "=" * 60)
    print(f"Q: {q}")
    answer, score = rag_pipeline(q)
    print(f"Reranker Score: {score:.4f}")
    display(Markdown(answer))


Q: Apa saja hak pekerja yang di-PHK menurut undang-undang?
Generated 2 hypothetical documents
  Hypothesis 1 (100 chars): Pekerja memiliki beberapa hak yang di-PHK menurut undang-undang Indonesia. Salah satunya adalah hak ...
  Hypothesis 2 (100 chars): Berdasarkan undang-undang dan konvensi Internasional tentang Pelecehan Kerja, hak-hak pekerja yang d...
Reranker Score: 1.1514


Hak pekerja yang di-PHK menurut undang-undang adalah hak atas program jaminan sosial. Hal ini mencakup hak atas upah minimum, tunjangan kesehatan, tunjangan pensiun, dan tunjangan kematian. Selain itu, pekerja juga memiliki hak atas hak istimewa seperti hak untuk masa percobaan kerja, masa kerja tetap, dan hak untuk meninggalkan pekerjaan jika mereka merasa tidak nyaman atau tidak puas dengan kondisi kerja.

---
**Sumber Referensi:**
  - PP Nomor 35 Tahun 2021.pdf (File: PP Nomor 35 Tahun 2021.pdf, Halaman: 3)
  - PP Nomor 35 Tahun 2021.pdf (File: PP Nomor 35 Tahun 2021.pdf, Halaman: 9)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 1094)


Q: Berapa lama masa percobaan kerja yang diperbolehkan?
Generated 2 hypothetical documents
  Hypothesis 1 (100 chars): Pemerintah Indonesia melarang menerapkan masa percobaan kerja yang lebih lama daripada 12 bulan dala...
  Hypothesis 2 (100 chars): Masa percobaan kerja yang diperbolehkan di Indonesia tidak memiliki definisi yang konsisten antara p...
Reranker Score: 2.7368


Berapa lama masa percobaan kerja yang diperbolehkan? 

Dalam dokumen tersebut, tidak ada informasi tentang masa percobaan kerja yang diperbolehkan.

---
**Sumber Referensi:**
  - PP Nomor 5 Tahun 2021.pdf (File: PP Nomor 5 Tahun 2021.pdf, Halaman: 334)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 1057)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 584)


Q: Bagaimana ketentuan pajak penghasilan untuk karyawan?
Generated 2 hypothetical documents
  Hypothesis 1 (100 chars): Pajak Penghasilan untuk karyawan adalah pajak yang diikuti oleh individu atau perusahaan untuk mengu...
  Hypothesis 2 (100 chars): Pajak penghasilan untuk karyawan dalam Indonesia dikenakan pada gaji yang dibayarkan dan dapat diken...
Reranker Score: 5.1527


Bagaimana ketentuan pajak penghasilan untuk karyawan? 

Untuk karyawan, pengenaan pajak tergantung pada jenis usaha mereka dan apakah mereka bekerja sebagai karyawan kontrak atau karyawan tetap. Untuk karyawan kontrak, mereka harus membayar pajak sebesar 20% dari penghasilan mereka setiap bulan. Untuk karyawan tetap, mereka harus membayar pajak sebesar 20% dari penghasilan mereka setiap tahun.

Untuk lebih detail, ada beberapa syarat dan kondisi yang harus dipenuhi oleh karyawan agar mereka bisa mendapatkan manfaat pajak. Misalnya, jika karyawan bekerja sebagai karyawan kontrak, mereka harus memiliki kontrak kerja yang valid dan telah dilakukan selama minimal satu bulan. Selain itu, mereka harus mencukupi minimum gaji yang disebutkan dalam kontrak kerja. Jika karyawan bekerja sebagai karyawan tetap, mereka harus memiliki kontrak kerja yang valid dan telah dilakukan selama minimal dua tahun. Selain itu, mereka harus mencukupi minimum gaji yang disebutkan dalam kontrak kerja.

Secara keseluruhan, pengenaan pajak untuk karyawan tergantung pada jenis usaha mereka dan apakah mereka bekerja sebagai karyawan kontrak atau karyawan tetap.

---
**Sumber Referensi:**
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 636)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 1060)
  - PP Nomor 5 Tahun 2021.pdf (File: PP Nomor 5 Tahun 2021.pdf, Halaman: 318)

## 17. Demo Interaktif RAG - Interactive Python Loop (Wajib)

Loop interaktif menggunakan `input()` agar pengguna dapat mengetikkan pertanyaan sendiri.
Ketik **"exit"** untuk keluar dari loop.


In [28]:
import warnings
warnings.filterwarnings("ignore")
warnings.showwarning = lambda *args, **kwargs: None
from IPython.display import display, Markdown

print("=" * 60)
print("  ASISTEN LEGAL AI - Tim Legal Perusahaan")
print("  Powered by Fine-tuned Qwen2.5-1.5B + RAG")
print("=" * 60)
print("Ketik pertanyaan Anda, atau ketik 'exit' untuk keluar.")
print()

counter = 0
while True:
    question = input("Pertanyaan Anda: ").strip()
    if question.lower() in ["exit", "quit", "keluar", ""]:
        print("\nTerima kasih telah menggunakan Asisten Legal AI!")
        break
    counter += 1
    print(f"\n{'='*60}")
    print(f"  Pertanyaan {counter}: {question}")
    print(f"{'='*60}")
    print("Sedang memproses...")
    answer, score = rag_pipeline(question)
    print(f"\nAsisten Legal AI (Relevance Score: {score:.4f}):")
    display(Markdown(answer))
    print()

print("\n" + "=" * 60)
print(f"Sesi interaktif selesai. Total {counter} pertanyaan dijawab.")


  ASISTEN LEGAL AI - Tim Legal Perusahaan
  Powered by Fine-tuned Qwen2.5-1.5B + RAG
Ketik pertanyaan Anda, atau ketik 'exit' untuk keluar.


  Pertanyaan 1: Berapa jam maksimal waktu kerja lembur dalam sehari dan dalam seminggu menurut peraturan yang berlaku?
Sedang memproses...
Generated 2 hypothetical documents
  Hypothesis 1 (100 chars): Dalam Peraturan Perundang-undangan Indonesia Nomor 7 Tahun 2009 tentang Konvensi Perangkat Negara (K...
  Hypothesis 2 (100 chars): Waktu kerja lembur sehari dalam negeri berlaku pada hukum perwakilan atau peraturan lokal. Namun, di...

Asisten Legal AI (Relevance Score: 0.7607):


Sebuah lembur adalah masa kerja yang tidak memiliki tujuan utama tertentu, seperti pekerjaan rutin atau aktivitas produktif. Namun, ada beberapa jenis lembur yang bisa diperlukan dalam bisnis, termasuk lembur tambahan, lembur harian, dan lembur mingguan.

Lembur tambahan biasanya digunakan untuk menjaga jalur produksi tetap berjalan, meningkatkan produktivitas, atau mengatasi masalah lain yang menyebabkan penundaan. Ini sering disebut sebagai "lembur tambahan" karena itu bukan lembur yang harus dilakukan secara rutin.

Lembur harian adalah lembur yang diberikan setiap hari, biasanya untuk mengisi waktu yang hilang atau untuk menjaga jalur produksi tetap berjalan. Ini juga bisa digunakan untuk mengatasi masalah lain yang menyebabkan penundaan.

Lembur mingguan adalah lembur yang diberikan setiap minggu, biasanya untuk mengisi waktu yang hilang atau untuk menjaga jalur produksi tetap berjalan. Ini juga bisa digunakan untuk mengatasi masalah lain yang menyebabkan penundaan.

Secara ringkas, jam maksimal waktu kerja lembur dalam sehari dan dalam seminggu akan bergantung pada jenis lembur yang diperlukan dan aturan perusahaan.

---
**Sumber Referensi:**
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 244)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 626)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 647)



  Pertanyaan 2: Apa saja kompensasi yang wajib diberikan perusahaan kepada pekerja yang terkena PHK?
Sedang memproses...
Generated 2 hypothetical documents
  Hypothesis 1 (100 chars): Pemerintah Indonesia telah mengatur penggunaan kompensasi yang wajib diberikan perusahaan kepada pek...
  Hypothesis 2 (100 chars): Kompensasi yang harus diberikan perusahaan kepada pekerja yang terkena PHK biasanya tergantung pada ...

Asisten Legal AI (Relevance Score: 5.1918):


Apakah ada kompensasi yang harus diberikan perusahaan kepada pekerja yang terkena PHK? 

Sebagai bahan referensi, mari kita lihat Perjanjian Kerja Waktu Tertentu yang Selanjutnya (PKWT). Menurut pasal 14, jika Pekerja/Buruh dinyatakan bersalah dalam kasus PHK, perusahaan harus memberikan uang penggantian hak sesuai ketentuan Pasal 40 ayat (4), serta uang penghargaan masa kerja sebesar 1 kali ketentuan Pasal 40 ayat (3).

Namun, jika Pekerja/Buruh dinyatakan tidak bersalah dalam kasus PHK, perusahaan harus memberikan uang penggantian hak sesuai ketentuan Pasal 40 ayat (4), serta uang penghargaan masa kerja sebesar 1 kali ketentuan Pasal 40 ayat (3).

---
**Sumber Referensi:**
  - PP Nomor 35 Tahun 2021.pdf (File: PP Nomor 35 Tahun 2021.pdf, Halaman: 3)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 684)
  - PP Nomor 35 Tahun 2021.pdf (File: PP Nomor 35 Tahun 2021.pdf, Halaman: 34)



  Pertanyaan 3: Berapa lama maksimal perjanjian kerja waktu tertentu (PKWT) dan apakah bisa diperpanjang?
Sedang memproses...
Generated 2 hypothetical documents
  Hypothesis 1 (100 chars): Sebagai pakar hukum ketenagakerjaan Indonesia, saya dapat mengkonfirmasikan bahwa maksimal perjanjia...
  Hypothesis 2 (100 chars): Perjanjian kerja waktu tertentu (PKWT) adalah kontrak yang dijalankan oleh karyawan untuk menghasilk...

Asisten Legal AI (Relevance Score: 5.2998):


...

Bagaimana ketentuan cuti tahunan dan cuti melahirkan bagi pekerja perempuan? 

Cuti tahunan dan cuti melahirkan adalah dua hal yang penting bagi pekerja perempuan, karena mereka memberikan kesempatan bagi wanita untuk mengambil masa liburan yang lebih banyak dan merencanakan kelahiran. Cuti tahunan biasanya digunakan oleh pekerja untuk menghabiskan waktu bersama keluarga, seperti liburan musim panas atau liburan Natal dan Dekade. Cuti melahirkan, di sisi lain, memberikan kesempatan bagi pekerja untuk menghabiskan waktu bersama anak-anak mereka.

Untuk membuat cuti tahunan dan cuti melahirkan lebih mudah bagi pekerja perempuan, beberapa negara telah mencoba melegalkan cuti tahunan dan cuti melahirkan. Misalnya, Amerika Serikat memiliki cuti tahunan yang disebut "Family and Medical Leave Act" (FMLA), yang memungkinkan pekerja untuk mengambil cuti selama periode tertentu jika mereka sedang sakit, hamil, menyusui, atau menjalani operasi medis. Selain itu, beberapa negara telah mencoba melegalkan cuti melahirkan, misalnya Inggris, Australia, dan Kanada, yang memberikan cuti kepada pekerja yang sedang menghadapi masalah kelahiran.

Meskipun beberapa negara telah berhasil melegalkan cuti tahunan dan cuti melahirkan, masih ada beberapa negara di mana cuti ini masih ilegal. Untuk contoh, cuti tahunan dan cuti melahirkan masih ilegal di China, India, dan Iran. Meskipun demikian, beberapa negara di mana cuti ini ilegal tetapi masih diperbolehkan bagi pekerja perempuan untuk mengambil cuti tahunan dan cuti melahirkan jika mereka membutuhkan.

Secara keseluruhan, cuti tahunan dan cuti melahirkan sangat penting bagi pekerja perempuan, karena memberikan kesempatan bagi mereka untuk menghabiskan waktu bersama keluarga dan merencanakan kelahiran. Meskipun masih ada beberapa negara di mana cuti ini ilegal, semakin banyak negara yang mulai melegalkan cuti ini, yang akan menjadi lebih baik bagi pekerja perempuan.

---
**Sumber Referensi:**
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 645)
  - PP Nomor 35 Tahun 2021.pdf (File: PP Nomor 35 Tahun 2021.pdf, Halaman: 42)
  - PP Nomor 5 Tahun 2021.pdf (File: PP Nomor 5 Tahun 2021.pdf, Halaman: 318)



  Pertanyaan 5: Bagaimana aturan pajak penghasilan PPh 21 terbaru untuk karyawan dengan gaji di atas 10 juta?
Sedang memproses...
Generated 2 hypothetical documents
  Hypothesis 1 (100 chars): Pajak penghasilan PPh 21 terbaru untuk karyawan dengan gaji di atas 10 juta diberlakukan pada tangga...
  Hypothesis 2 (100 chars): Aturan pajak penghasilan PPh 21 terbaru untuk karyawan dengan gaji di atas 10 juta yang berlaku di I...

Asisten Legal AI (Relevance Score: 5.3527):


Aturan pajak penghasilan PPh 21 terbaru untuk karyawan dengan gaji di atas 10 juta adalah 20%. Ini berarti bahwa setiap karyawan dengan gaji di atas 10 juta akan menerima tarif pajak sebesar 20% dari penghasilan mereka.

Untuk contoh, jika seorang karyawan dengan gaji $120,000 mendapatkan PPh 21, maka ia akan menerima tarif pajak sebesar $24,000 ($120,000 x 20%).

Namun, jika seorang karyawan dengan gaji di bawah atau sama dengan $10 juta, maka ia tidak menerima tarif pajak sebesar 20%, karena ada batasan pajak tertentu. Sebagai contoh, jika seorang karyawan dengan gaji $99,999 mendapatkan PPh 21, maka ia hanya menerima tarif pajak sebesar 19.999 ($99,999 x 20%).

---
**Sumber Referensi:**
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 636)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 169)
  - UU Nomor 6 Tahun 2023.pdf (File: UU Nomor 6 Tahun 2023.pdf, Halaman: 662)



Terima kasih telah menggunakan Asisten Legal AI!

Sesi interaktif selesai. Total 5 pertanyaan dijawab.


Generated 2 hypothetical documents
  Hypothesis 1 (100 chars): Dalam peraturan perundang-undangan Indonesia, berapa jam maksimal waktu kerja lembur dalam sehari da...
  Hypothesis 2 (100 chars): Pemerintah Indonesia memiliki peraturan yang membatasi jam waktu kerja lembur dalam sehari dan dalam...


## 18. Interface Gradio (Opsional)

In [26]:
import warnings
warnings.filterwarnings("ignore")
warnings.showwarning = lambda *args, **kwargs: None

try:
    import gradio as gr

    def gradio_rag(question, use_hyde, threshold):
        try:
            answer, score = rag_pipeline(
                question, use_hyde=use_hyde,
                rerank_threshold=threshold
            )
            return f"**Score:** {score:.4f}\n\n{answer}"
        except Exception as e:
            return f"Error: {str(e)}"

    demo = gr.Interface(
        fn=gradio_rag,
        inputs=[
            gr.Textbox(label="Pertanyaan", lines=3),
            gr.Checkbox(label="Gunakan HyDE", value=True),
            gr.Slider(0.0, 1.0, 0.1, 0.05, label="Threshold"),
        ],
        outputs=gr.Markdown(label="Jawaban"),
        title="Asisten Legal AI",
    )
    demo.launch(share=True)
except KeyboardInterrupt:
    print("Gradio server dihentikan.")
except Exception as e:
    print(f"Gradio tidak tersedia: {e}")
    print("Langkah ini opsional.")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9d3450a167151f6882.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Generated 2 hypothetical documents
  Hypothesis 1 (100 chars): Tata cara kerja lembur dapat berbeda-beda dalam berbagai perusahaan dan industri, dan peraturan yang...
  Hypothesis 2 (100 chars): Menurut peraturan perijinan dan kerja yang berlaku di Indonesia, lembur dapat diatur oleh perusahaan...
